# Practice: What Distinguishes Health Coverage, Robustly?

**Computational Text Analysis Pack · Module 04 — Comparing Corpora**

*Technique:* Keyness with effect size, dispersion and a robustness check, end to end

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/03-comparing-corpora/04e_practice_comparing.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## The question

> **Which words genuinely characterise health reporting in this corpus, as
> opposed to appearing at the top of a keyness list by accident?**

Modules 4a–4d gave the pieces: log-likelihood, effect size, the significance
critique. This assembles them into one defensible procedure.

### How to use this notebook

This is a worked case, not a tutorial. Run it once to see the shape of the
argument, then change the question at the top to one of your own and run it
again. The cells are written so that swapping the question is the only edit you
need.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
import numpy as np
target_docs = [str(t) for t in df[df["category"] == "health"]["text"]]
ref_docs = [str(t) for t in df[df["category"] != "health"]["text"]]
ca, cb = Counter(), Counter()
for t in target_docs: ca.update(tokenise(t))
for t in ref_docs: cb.update(tokenise(t))
na, nb = sum(ca.values()), sum(cb.values())
print(f"target {len(target_docs)} docs / {na:,} tokens")
print(f"reference {len(ref_docs)} docs / {nb:,} tokens")

def ll(a, b):
    if a == 0 and b == 0: return 0.0
    ea, eb = na*(a+b)/(na+nb), nb*(a+b)/(na+nb)
    s = 0.0
    if a: s += a*np.log(a/ea)
    if b: s += b*np.log(b/eb)
    return 2*s
def log_ratio(a, b):
    return np.log2(((a+0.5)/na) / ((b+0.5)/nb))

rows = [{"word": w, "target": ca[w], "ref": cb[w], "LL": ll(ca[w], cb[w]),
         "log_ratio": log_ratio(ca[w], cb[w])}
        for w in ca if ca[w] + cb[w] >= 25]
key = pd.DataFrame(rows).sort_values("LL", ascending=False)
print("\nTop 12 by log-likelihood alone:")
print(key.head(12).to_string(index=False, float_format=lambda v: f"{v:.2f}"))

In [ ]:
# Filter 1: effect size. Significance without it is just a corpus-size report.
strong = key[(key["LL"] >= 15.13) & (key["log_ratio"] >= 1.0)]
print(f"{len(key)} candidates -> {len(strong)} pass LL>=15.13 AND log-ratio>=1.0\n")
print(strong.head(12).to_string(index=False, float_format=lambda v: f"{v:.2f}"))

In [ ]:
# Filter 2: dispersion. A word concentrated in three articles is not
# characteristic of health coverage; it is characteristic of three articles.
def gries_dp(word, docs, n_parts=12):
    size = max(len(docs)//n_parts, 1)
    parts = [docs[i*size:(i+1)*size] for i in range(n_parts)]
    freqs = np.array([sum(tokenise(d).count(word) for d in p) for p in parts], float)
    if freqs.sum() == 0: return 1.0
    sizes = np.array([sum(len(tokenise(d)) for d in p) for p in parts], float)
    return float(np.abs(freqs/freqs.sum() - sizes/sizes.sum()).sum()/2)

check = strong.head(15).copy()
check["DP"] = [gries_dp(w, target_docs[:600]) for w in check["word"]]
check["verdict"] = np.where(check["DP"] < 0.35, "spread - keep", "clumped - suspect")
print(check[["word", "target", "LL", "log_ratio", "DP", "verdict"]]
      .to_string(index=False, float_format=lambda v: f"{v:.2f}"))

In [ ]:
# Filter 3: does it survive resampling? (module 11c's test, applied here)
rng = np.random.default_rng(4)
survive = Counter()
final = set(check.loc[check["verdict"].str.startswith("spread"), "word"])
for _ in range(10):
    idx = rng.integers(0, len(target_docs), len(target_docs))
    c2 = Counter()
    for i in idx: c2.update(tokenise(target_docs[i]))
    n2 = sum(c2.values())
    sc = {w: np.log2(((c2[w]+.5)/n2)/((cb[w]+.5)/nb)) for w in check["word"]}
    for w in sorted(sc, key=sc.get, reverse=True)[:8]:
        survive[w] += 1
print("survival across 10 bootstrap resamples (top-8 each time):\n")
for w in check["word"]:
    bar = "#" * survive[w]
    print(f"   {w:<18}{survive[w]:>3}/10  {bar}")
print()
print("The words that pass ALL THREE filters are the ones to report:")
robust = [w for w in final if survive[w] >= 7]
print("   " + (", ".join(sorted(robust)) if robust else "(none survived - tighten or widen)"))

## What this licenses

**Defensible:** "The following words characterise health coverage in this
corpus: they are statistically distinguished from the reference corpus, have a
log-ratio above 1, are evenly dispersed across the subcorpus, and remain in the
top eight under bootstrap resampling."

That sentence is long, and its length is the point. Each clause blocks a
specific failure: corpus-size inflation, a large p on a trivial difference, a
word from three articles, and an unstable ranking.

**Not defensible:** the top-12 log-likelihood list you started with. Compare it
against the final list — several words did not survive, and reporting them would
have been reporting noise.

### 🔧 Your turn 1

Run the whole procedure for a different category and count how many of the original top 12 survive all three filters.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Survival rates vary a lot by category, and the pattern is informative: narrow,
technical categories keep most of their keywords, while broad ones like
'politics' lose many, because their vocabulary overlaps everything else. If
almost nothing survives, that is a real result — the category is not lexically
distinctive in this corpus — and it is a better finding than a list of words
that would not replicate.

</details>